# Notebook 01 — Data Exploration

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 01 of 11 — exploratory data analysis

---

### Purpose

This notebook establishes that the dataset is fit for the research claims made in the dissertation. It is not a generic EDA. Every section below answers a question that a later chapter depends on:

| Section | Question answered | Feeds into |
|---|---|---|
| 1 | Is the label file internally consistent and trustworthy? | Ch. 3 validity, Ch. 5 dataset description |
| 2 | How are the four classes and three age classes distributed, and are they associated? | Ch. 4 class design, **Ch. 5 metadata-fusion ablation** |
| 3 | Is there evidence of annotation-order or annotator-effort bias? | Ch. 5 threats to validity |
| 4 | Do image capture conditions differ systematically between classes (i.e. is there a shortcut the model could exploit)? | **Ch. 5 threats to validity — leakage** |
| 5 | Are the three morphology classes geometrically real, or only labels? | **Ch. 4 justification of the morphology taxonomy**, Ch. 5 geometric baseline |
| 6 | What input resolution preserves crack evidence? | Ch. 4 pre-processing decision, nb 02 |
| 7 | What do the classes actually look like? | Ch. 4 figures |
| 8 | A single clean master table for every downstream notebook | nb 02–11 |

### Outputs written
```
results/01_data_exploration/figures/*.png     publication-quality, 300 dpi
results/01_data_exploration/tables/*.csv      every table reported in the thesis
results/01_data_exploration/eda_summary.json  machine-readable digest
data/processed/master_table.csv               the single source of truth for nb 02+
```

### Note on GPU
This stage is I/O- and CPU-bound (JPEG decode, OpenCV). It is parallelised with `joblib` across CPU cores; GPU acceleration begins in notebook 03 (augmentation) and 05 (model selection). The environment report below confirms the CUDA device is visible so that later stages run without surprises.

---
## 0. Configuration and environment

In [ ]:
# --- EDIT HERE IF YOUR FOLDERS DIFFER -------------------------------------
# The notebook expects the repository layout:
#   Artefact/notebooks/01_data_exploration.ipynb   <- you are here
#   Artefact/data/annotations/final_labels_v3.csv
#   Artefact/data/images/cropped/E001.jpg ... E652.jpg
#
# If your cropped images live somewhere else, set the path below (or set the
# environment variable EGG_IMAGES_DIR before launching Jupyter).
import os
os.environ["EGG_IMAGES_DIR"] = r"E:\Degree\Final Research\Egg_Crack_Dataset\Eggshell-Diagnostic-Research\Artefact\data\images\cropped"      # <- uncomment & edit if needed
os.environ["EGG_ARTEFACT_ROOT"] = r"E:\Degree\Final Research\Egg_Crack_Dataset\Eggshell-Diagnostic-Research\Artefact"  # <- uncomment & edit if needed

N_JOBS = -1          # joblib workers for image statistics (-1 = all cores)
SAMPLE_GRID_N = 8    # images per class in the sample grids
FIG_DPI = 300

In [ ]:
import sys, json, math, platform, warnings
from pathlib import Path
from datetime import datetime

# make Artefact/src importable from Artefact/notebooks
_here = Path.cwd()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _src is None:
    raise RuntimeError(
        "Could not locate Artefact/src/common. Run this notebook from Artefact/notebooks/ "
        "or set EGG_ARTEFACT_ROOT."
    )
sys.path.insert(0, str(_src))

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats
from joblib import Parallel, delayed

from common import project_config as CFG

warnings.filterwarnings("ignore", category=FutureWarning)
CFG.seed_everything()

print("=" * 72)
print("ENVIRONMENT")
print("=" * 72)
print(f"python      {platform.python_version()}  ({platform.system()} {platform.release()})")
print(f"numpy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"opencv      {cv2.__version__}")
print(f"scipy       {scipy.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"cpu cores   {os.cpu_count()}")
try:
    import torch
    print(f"torch       {torch.__version__}  cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu         {torch.cuda.get_device_name(0)}  "
              f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
        print(f"cuda        {torch.version.cuda}  cudnn={torch.backends.cudnn.version()}")
    else:
        print("gpu         NOT VISIBLE -- fine for notebook 01, but fix before notebook 05")
except ImportError:
    print("torch       not installed (not needed for notebook 01)")
print()
print("=" * 72)
print("PATHS")
print("=" * 72)
print(CFG.describe())

In [ ]:
# --- output locations -----------------------------------------------------
STAGE = "01_data_exploration"
OUT = CFG.results_dir(STAGE)
FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PROC_DIR = CFG.DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# --- consistent publication style for every figure in the dissertation ----
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": FIG_DPI,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.facecolor": "white", "savefig.facecolor": "white",
})

def boxplot(ax, data, names, **kw):
    """matplotlib renamed boxplot(labels=) to tick_labels= in 3.9; stay compatible."""
    try:
        return ax.boxplot(data, tick_labels=names, **kw)
    except TypeError:
        return ax.boxplot(data, labels=names, **kw)

_FIG_N = {"i": 0}
FIGURE_INDEX, TABLE_INDEX = [], []

def savefig(fig, name, caption="", dpi=None):
    """Save at publication dpi and record the caption for the figure list.

    Photographic grids are saved lower: at 300 dpi they run to several MB each,
    which bloats the git repository for no visible gain on a printed page."""
    _FIG_N["i"] += 1
    fname = f"{_FIG_N['i']:02d}_{name}.png"
    fig.savefig(FIG_DIR / fname, bbox_inches="tight", dpi=dpi or FIG_DPI)
    FIGURE_INDEX.append({"stage": STAGE, "file": fname, "caption": caption})
    print(f"  [figure] {fname}")

def savetable(df, name, caption=""):
    """Save a table as CSV and record it."""
    p = TAB_DIR / f"{name}.csv"
    df.to_csv(p, index=True)
    TABLE_INDEX.append({"stage": STAGE, "file": p.name, "caption": caption,
                        "shape": list(df.shape)})
    print(f"  [table ] {p.name}  {df.shape}")
    return df

SUMMARY = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds")}
print(f"figures -> {FIG_DIR}\ntables  -> {TAB_DIR}\nprocessed -> {PROC_DIR}")

---
## 1. Label file integrity

The annotation was produced by a single annotator in Label Studio and verified by the
collaborating hatchery manager. Before any modelling claim is made, the file must be shown to be
internally consistent: the decision tree used during annotation implies hard constraints
(an *Intact* egg can have no pattern, no polygon and no root-cause split; a *Micro_cracked* egg
must have all three). Section 5.8 of the dissertation reports the result of this audit.

In [ ]:
PATTERNS, LABEL4, AGE_ORDER = CFG.PATTERNS, CFG.LABEL4, CFG.AGE_ORDER

def load_labels(csv_path: Path) -> pd.DataFrame:
    """Read final_labels_v3.csv and derive the modelling columns."""
    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    # Label Studio stores '/data/upload/1/<random>-E001.jpg'; the real egg id is the suffix
    df["egg_id"] = df["image"].str.extract(r"-(E\d{3})\.jpg", expand=False)

    # four-class target used by formulation (b) and the multi-task head
    df["label4"] = np.where(df["egg_status"].eq("Intact"), "Intact", df["crack_pattern"])
    df["is_cracked"] = df["egg_status"].eq("Micro_cracked").astype(int)

    # number of separate crack polygons drawn on this egg
    df["n_polygons"] = df["crack_region"].apply(
        lambda s: 0 if not isinstance(s, str) or not s.strip() else len(json.loads(s)))

    for c in ("created_at", "updated_at"):
        df[c] = pd.to_datetime(df[c], format="ISO8601", utc=True)
    # Label Studio timestamps are UTC; the farm operates on Sri Lanka time (UTC+5:30)
    df["created_local"] = df["created_at"].dt.tz_convert("Asia/Colombo")
    df["annotation_minutes"] = df["lead_time"] / 60.0

    df["age_class"] = pd.Categorical(df["age_class"], categories=AGE_ORDER, ordered=True)
    df["label4"] = pd.Categorical(df["label4"], categories=LABEL4, ordered=True)
    df["crack_pattern_cat"] = pd.Categorical(df["crack_pattern"], categories=PATTERNS, ordered=True)
    return df.sort_values("egg_id").reset_index(drop=True)

labels = load_labels(CFG.LABELS_CSV)
print(f"loaded {len(labels)} rows x {labels.shape[1]} columns from {CFG.LABELS_CSV.name}")
labels[["egg_id", "egg_status", "crack_pattern", "age_class", "label4",
        "n_polygons", "physiological_nutritional_pct", "lead_time"]].head(8)

In [ ]:
# --- schema and missingness audit ----------------------------------------
schema = pd.DataFrame({
    "dtype": labels.dtypes.astype(str),
    "n_missing": labels.isna().sum(),
    "pct_missing": (labels.isna().mean() * 100).round(1),
    "n_unique": labels.nunique(dropna=True),
    "example": [labels[c].dropna().iloc[0] if labels[c].notna().any() else None
                for c in labels.columns],
})
schema["example"] = schema["example"].astype(str).str.slice(0, 58)
savetable(schema, "t01_schema_audit",
          "Schema and missingness audit of final_labels_v3.csv")
schema

In [ ]:
def validate_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Constraints implied by the annotation decision tree. All must pass."""
    intact = df["egg_status"].eq("Intact")
    crk = df["egg_status"].eq("Micro_cracked")
    pct_sum = df["physiological_nutritional_pct"] + df["mechanical_environmental_pct"]
    checks = [
        ("row count equals 652 eggs", len(df) == 652, len(df)),
        ("egg_id parsed for every row", df["egg_id"].notna().all(), int(df["egg_id"].isna().sum())),
        ("egg_id unique", df["egg_id"].is_unique, int(df["egg_id"].duplicated().sum())),
        ("egg_id covers E001..E652 with no gaps",
         sorted(df["egg_id"]) == [f"E{i:03d}" for i in range(1, len(df) + 1)],
         len(set(f"E{i:03d}" for i in range(1, 653)) - set(df["egg_id"]))),
        ("annotation_id unique", df["annotation_id"].is_unique,
         int(df["annotation_id"].duplicated().sum())),
        ("egg_status vocabulary closed", set(df["egg_status"]) <= set(CFG.STATUS),
         sorted(set(df["egg_status"]))),
        ("crack_pattern vocabulary closed",
         set(df["crack_pattern"].dropna()) <= set(PATTERNS), sorted(set(df["crack_pattern"].dropna()))),
        ("age_class complete and closed",
         df["age_class"].notna().all() and set(df["age_class"].dropna()) <= set(AGE_ORDER),
         int(df["age_class"].isna().sum())),
        ("Intact -> crack_pattern empty", df.loc[intact, "crack_pattern"].isna().all(),
         int(df.loc[intact, "crack_pattern"].notna().sum())),
        ("Intact -> no polygon drawn", df.loc[intact, "n_polygons"].eq(0).all(),
         int(df.loc[intact, "n_polygons"].gt(0).sum())),
        ("Intact -> no root-cause split", df.loc[intact, "physiological_nutritional_pct"].isna().all(),
         int(df.loc[intact, "physiological_nutritional_pct"].notna().sum())),
        ("Micro_cracked -> crack_pattern present", df.loc[crk, "crack_pattern"].notna().all(),
         int(df.loc[crk, "crack_pattern"].isna().sum())),
        ("Micro_cracked -> at least one polygon", df.loc[crk, "n_polygons"].ge(1).all(),
         int(df.loc[crk, "n_polygons"].lt(1).sum())),
        ("Micro_cracked -> root-cause split present",
         df.loc[crk, "physiological_nutritional_pct"].notna().all(),
         int(df.loc[crk, "physiological_nutritional_pct"].isna().sum())),
        ("phys% + mech% == 100 for every cracked egg",
         bool(np.allclose(pct_sum[crk].to_numpy(dtype=float), 100.0)),
         sorted(pd.unique(pct_sum[crk].dropna()).tolist())),
        ("single annotator (inter-rater reliability not estimable)",
         df["annotator"].nunique() == 1, int(df["annotator"].nunique())),
        ("lead_time strictly positive", (df["lead_time"] > 0).all(), float(df["lead_time"].min())),
        ("updated_at >= created_at", (df["updated_at"] >= df["created_at"]).all(),
         int((df["updated_at"] < df["created_at"]).sum())),
    ]
    out = pd.DataFrame(checks, columns=["constraint", "passed", "observed"])
    out["observed"] = out["observed"].astype(str)
    return out

integrity = validate_labels(labels)
savetable(integrity.set_index("constraint"), "t02_label_integrity",
          "Label-file integrity audit against the annotation decision tree")
n_fail = int((~integrity["passed"]).sum())
SUMMARY["integrity_checks_total"] = len(integrity)
SUMMARY["integrity_checks_failed"] = n_fail
print(f"\n{len(integrity) - n_fail}/{len(integrity)} constraints satisfied"
      + ("" if n_fail == 0 else "  <-- INVESTIGATE THE FAILURES BELOW"))
integrity.style.map(lambda v: "background-color:#d7f0d7" if v is True
                    else ("background-color:#f7d4d4" if v is False else ""), subset=["passed"])

---
## 2. Class structure and the metadata question

Three things matter here.

1. **The four-class target.** The novelty of this work is that the model predicts
   *Intact / Hairline / Star / Network* rather than the binary intact–damaged decision used by
   Cengel et al. (2025) and Cheng et al. (2024). The class sizes determine how imbalance must be
   handled in notebook 05.
2. **The age-class distribution.** Breeder age is the biological metadata that makes root-cause
   inference possible.
3. **Whether pattern and age are statistically associated.** This is the single most important
   number in this notebook. If crack pattern is *independent* of age class, then feeding age into
   the image classifier cannot improve pattern prediction, and the correct architecture fuses age
   at the *decision* layer rather than the feature layer. The ablation in notebook 06 tests this
   empirically; this cell predicts what it will find.

In [ ]:
def balance_table(s: pd.Series, name: str) -> pd.DataFrame:
    vc = s.value_counts(dropna=False).rename("n").to_frame()
    vc["pct"] = (vc["n"] / vc["n"].sum() * 100).round(1)
    vc.index.name = name
    return vc

bal_status = balance_table(labels["egg_status"], "egg_status")
bal_label4 = balance_table(labels["label4"], "label4")
bal_age = balance_table(labels["age_class"], "age_class")

for t, nm in [(bal_status, "status"), (bal_label4, "label4"), (bal_age, "age_class")]:
    print(f"\n--- {nm} ---"); print(t.to_string())

savetable(bal_label4, "t03_class_balance_label4", "Four-class target distribution")
savetable(bal_age, "t04_age_class_balance", "Breeder age-class distribution")

# imbalance ratio drives the class-weighting decision in notebook 05
ir = bal_label4["n"].max() / bal_label4["n"].min()
SUMMARY["n_eggs"] = int(len(labels))
SUMMARY["class_counts_label4"] = bal_label4["n"].to_dict()
SUMMARY["class_counts_age"] = bal_age["n"].to_dict()
SUMMARY["imbalance_ratio_label4"] = round(float(ir), 2)
print(f"\nimbalance ratio (largest / smallest class) = {ir:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

ax = axes[0]
d = bal_status.reindex(CFG.STATUS)
ax.bar(d.index, d["n"], color=["#4C72B0", "#C44E52"], width=0.6)
for i, (n, p) in enumerate(zip(d["n"], d["pct"])):
    ax.text(i, n + 6, f"{n}\n{p:.1f}%", ha="center", fontsize=9)
ax.set_title("(a) Detection target"); ax.set_ylabel("eggs"); ax.set_ylim(0, 400)

ax = axes[1]
d = bal_label4.reindex(LABEL4)
ax.bar(d.index, d["n"], color=[CFG.PALETTE[c] for c in d.index], width=0.65)
for i, (n, p) in enumerate(zip(d["n"], d["pct"])):
    ax.text(i, n + 6, f"{n}\n{p:.1f}%", ha="center", fontsize=9)
ax.set_title("(b) Four-class morphology target"); ax.set_ylim(0, 400)
ax.tick_params(axis="x", labelrotation=0)

ax = axes[2]
d = bal_age.reindex(AGE_ORDER)
ax.bar(d.index, d["n"], color=[CFG.AGE_PALETTE[c] for c in d.index], width=0.6)
for i, (n, p) in enumerate(zip(d["n"], d["pct"])):
    ax.text(i, n + 4, f"{n}\n{p:.1f}%", ha="center", fontsize=9)
ax.set_title("(c) Breeder age class"); ax.set_ylim(0, 280)
ax.set_xticks(range(3)); ax.set_xticklabels(["Early\nlay", "Peak\nproduction", "Late\nlay"])

fig.suptitle("Dataset composition (n = 652 candled hatching eggs)", y=1.04, fontsize=12)
fig.tight_layout()
savefig(fig, "class_balance",
        "Distribution of the detection target, the four-class morphology target and breeder age class")
plt.show()

In [ ]:
def association(ct: pd.DataFrame, label: str) -> dict:
    """Chi-square test of independence with Cramer's V effect size."""
    chi2, p, dof, exp = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = math.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    min_exp = float(exp.min())
    res = {"comparison": label, "chi2": round(float(chi2), 3), "dof": int(dof),
           "p_value": round(float(p), 4), "cramers_v": round(float(v), 3),
           "n": int(n), "min_expected_count": round(min_exp, 1),
           "assumption_ok": bool(min_exp >= 5)}
    return res

ct_pattern_age = pd.crosstab(labels["crack_pattern_cat"], labels["age_class"])
ct_status_age = pd.crosstab(labels["egg_status"], labels["age_class"])
ct_label4_age = pd.crosstab(labels["label4"], labels["age_class"])

assoc = pd.DataFrame([
    association(ct_pattern_age, "crack pattern x age class (cracked eggs only)"),
    association(ct_status_age, "egg status x age class (all eggs)"),
    association(ct_label4_age, "four-class label x age class (all eggs)"),
]).set_index("comparison")

print(ct_pattern_age.to_string(), "\n")
print("row % within pattern:")
print((ct_pattern_age.div(ct_pattern_age.sum(1), axis=0) * 100).round(1).to_string(), "\n")
savetable(ct_pattern_age, "t05_crosstab_pattern_x_age", "Crack pattern by breeder age class")
savetable(assoc, "t06_association_tests",
          "Chi-square tests of independence between labels and breeder age class")
SUMMARY["association_tests"] = assoc.reset_index().to_dict("records")
assoc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))

# (a) grouped counts
ax = axes[0]
x = np.arange(len(PATTERNS)); w = 0.26
for k, age in enumerate(AGE_ORDER):
    vals = [ct_pattern_age.loc[p, age] for p in PATTERNS]
    ax.bar(x + (k - 1) * w, vals, w, label=age, color=CFG.AGE_PALETTE[age])
    for xi, v in zip(x + (k - 1) * w, vals):
        ax.text(xi, v + 1, str(v), ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(PATTERNS)
ax.set_ylabel("eggs"); ax.set_title("(a) Crack pattern by breeder age class")
ax.legend(fontsize=9)

# (b) standardised residuals: where does the observed table depart from independence?
chi2, p, dof, exp = stats.chi2_contingency(ct_pattern_age)
resid = (ct_pattern_age.values - exp) / np.sqrt(exp)
ax = axes[1]
im = ax.imshow(resid, cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(3)); ax.set_xticklabels([a.replace(" ", "\n") for a in AGE_ORDER], fontsize=9)
ax.set_yticks(range(3)); ax.set_yticklabels(PATTERNS)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{resid[i, j]:+.2f}", ha="center", va="center", fontsize=9)
ax.set_title(f"(b) Pearson residuals\n$\\chi^2$({dof}) = {chi2:.2f}, p = {p:.3f}, "
             f"V = {math.sqrt(chi2/(ct_pattern_age.values.sum()*2)):.3f}")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="standardised residual")

fig.tight_layout()
savefig(fig, "pattern_by_age_association",
        "Association between crack morphology and breeder age class, with Pearson residuals")
plt.show()

print("\nInterpretation guide: |residual| < 2 everywhere and p > 0.05 means pattern and age are")
print("statistically independent in this sample. If so, age carries no information about which")
print("pattern appears, and the honest architectural conclusion is that age belongs in the")
print("root-cause decision layer, not in the image feature fusion. Notebook 06 tests this.")

In [ ]:
# --- recover the root-cause rule base from the labelled data --------------
# physiological/mechanical split should be a pure function of (pattern, age class).
sub = labels.dropna(subset=["crack_pattern"])
rules = (sub.groupby(["crack_pattern_cat", "age_class"], observed=True)
            .agg(n=("physiological_nutritional_pct", "size"),
                 phys_pct=("physiological_nutritional_pct", "first"),
                 mech_pct=("mechanical_environmental_pct", "first"),
                 n_distinct_phys=("physiological_nutritional_pct", "nunique"),
                 n_distinct_interpretation=("interpretation", "nunique"),
                 n_distinct_actions=("suggested_actions", "nunique"))
            .reset_index()
            .rename(columns={"crack_pattern_cat": "crack_pattern"}))
rules["deterministic"] = (rules[["n_distinct_phys", "n_distinct_interpretation",
                                 "n_distinct_actions"]] == 1).all(axis=1)
savetable(rules.set_index(["crack_pattern", "age_class"]), "t07_recovered_root_cause_rules",
          "Root-cause rule base recovered from the labelled data (pattern x age -> phys/mech split)")
SUMMARY["root_cause_rules_deterministic"] = bool(rules["deterministic"].all())
SUMMARY["root_cause_rules"] = rules[["crack_pattern", "age_class", "phys_pct", "mech_pct"]]\
    .astype({"crack_pattern": str, "age_class": str}).to_dict("records")

print(rules.to_string(index=False))
print(f"\nrule base is deterministic: {rules['deterministic'].all()}")
print(f"distinct interpretation texts in the whole file: {labels['interpretation'].nunique()}")
print(f"distinct suggested-action texts in the whole file: {labels['suggested_actions'].nunique()}")

# heatmap of the physiological share
piv = rules.pivot(index="crack_pattern", columns="age_class", values="phys_pct").reindex(PATTERNS)[AGE_ORDER]
fig, ax = plt.subplots(figsize=(5.6, 3.4))
im = ax.imshow(piv.values, cmap="YlOrRd", vmin=0, vmax=100)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f"{piv.values[i, j]:.0f}%", ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="white" if piv.values[i, j] > 55 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels([a.replace(" ", "\n") for a in AGE_ORDER])
ax.set_yticks(range(3)); ax.set_yticklabels(PATTERNS)
ax.set_title("Physiological / nutritional share of attributed cause")
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.85, label="% physiological")
fig.tight_layout()
savefig(fig, "root_cause_rule_base",
        "Expert rule base mapping crack morphology and breeder age class to the physiological share of cause")
plt.show()

---
## 3. Annotation process: effort, order and potential bias

`lead_time` is the total seconds the annotator spent on each egg. It is a proxy for how difficult
that egg was to classify, and it supports two claims in Chapter 5: that the classes differ in
intrinsic difficulty, and that there is no systematic drift in how eggs were labelled over the
course of the annotation session. A strong correlation between Label Studio task order and class
would indicate the eggs were presented in blocks, which would weaken the independence assumption
behind the train/test split.

In [ ]:
eff = (labels.groupby("label4", observed=True)["lead_time"]
       .agg(n="size", median_s="median", mean_s="mean", sd_s="std",
            p90_s=lambda s: s.quantile(0.90), max_s="max").round(1))
savetable(eff, "t08_annotation_effort",
          "Annotation time per egg by class (proxy for labelling difficulty)")
print(eff.to_string())

kw = stats.kruskal(*[g["lead_time"].values for _, g in labels.groupby("label4", observed=True)])
print(f"\nKruskal-Wallis across the four classes: H = {kw.statistic:.2f}, p = {kw.pvalue:.4g}")

# order effect: does task id (upload order) predict the class?
order_tab = pd.crosstab(pd.qcut(labels["id"], 4, labels=["Q1", "Q2", "Q3", "Q4"]), labels["label4"])
chi2_o, p_o, dof_o, _ = stats.chi2_contingency(order_tab)
print(f"class x annotation-order quartile: chi2({dof_o}) = {chi2_o:.2f}, p = {p_o:.4f}")
SUMMARY["annotation_effort_median_s"] = eff["median_s"].to_dict()
SUMMARY["kruskal_leadtime_p"] = float(kw.pvalue)
SUMMARY["order_effect_p"] = float(p_o)
savetable(order_tab, "t09_class_by_annotation_order",
          "Class distribution across quartiles of annotation order (order-bias check)")
order_tab

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

ax = axes[0]
data = [labels.loc[labels["label4"].eq(c), "lead_time"].values for c in LABEL4]
bp = boxplot(ax, data, LABEL4, patch_artist=True, showfliers=False, widths=0.55)
for patch, c in zip(bp["boxes"], LABEL4):
    patch.set_facecolor(CFG.PALETTE[c]); patch.set_alpha(0.75)
ax.set_yscale("log"); ax.set_ylabel("seconds per egg (log scale)")
ax.set_title(f"(a) Annotation effort by class\nKruskal-Wallis p = {kw.pvalue:.3g}")

ax = axes[1]
for c in LABEL4:
    s = labels.loc[labels["label4"].eq(c)].sort_values("id")
    ax.scatter(s["id"], s["lead_time"], s=9, alpha=0.6, color=CFG.PALETTE[c], label=c)
ax.set_yscale("log"); ax.set_xlabel("Label Studio task id (annotation order)")
ax.set_ylabel("seconds"); ax.set_title("(b) Effort over the annotation session")
ax.legend(fontsize=8, ncol=2)

ax = axes[2]
prop = order_tab.div(order_tab.sum(1), axis=0) * 100
bottom = np.zeros(len(prop))
for c in LABEL4:
    ax.bar(prop.index.astype(str), prop[c], bottom=bottom, color=CFG.PALETTE[c], label=c, width=0.7)
    bottom += prop[c].values
ax.set_ylabel("% of eggs"); ax.set_xlabel("quartile of annotation order")
ax.set_title(f"(c) Class mix is stable over time\n$\\chi^2$ p = {p_o:.3f}")
ax.legend(fontsize=8, ncol=2, loc="lower right")

fig.tight_layout()
savefig(fig, "annotation_process",
        "Annotation effort by class and stability of the class mix across the annotation session")
plt.show()

---
## 4. Image inventory, capture conditions and confound screening

This section does the work that protects the headline result. Candled images are produced by a
smartphone over a transillumination source in a light-proof box, which means brightness, exposure
and focus vary between captures. If any of those capture variables differ systematically between
classes, a convolutional network can reach a high score by reading the capture condition rather
than the crack — a shortcut that would invalidate the contribution. The Kruskal–Wallis tests below
screen for exactly that, and the result is reported as a threat to validity in Section 5.8.

In [ ]:
# --- locate the images and reconcile them with the label file -------------
if not CFG.IMAGES_DIR.is_dir():
    raise FileNotFoundError(
        f"IMAGES_DIR does not exist: {CFG.IMAGES_DIR}\n"
        f"Set os.environ['EGG_IMAGES_DIR'] in the configuration cell at the top."
    )

files = sorted(CFG.IMAGES_DIR.glob(CFG.IMAGE_GLOB))
found = pd.DataFrame({"egg_id": [f.stem for f in files], "path": [str(f) for f in files]})
print(f"{len(found)} image files in {CFG.IMAGES_DIR}")

missing = sorted(set(labels["egg_id"]) - set(found["egg_id"]))
extra = sorted(set(found["egg_id"]) - set(labels["egg_id"]))
print(f"labelled but no image file : {len(missing)}  {missing[:10]}")
print(f"image file but not labelled: {len(extra)}  {extra[:10]}")
if missing:
    raise FileNotFoundError(
        f"{len(missing)} labelled eggs have no image. Check the file names are exactly "
        f"E001.jpg ... E652.jpg in {CFG.IMAGES_DIR}."
    )
SUMMARY["n_image_files"] = int(len(found))
SUMMARY["n_missing_images"] = len(missing)
SUMMARY["n_unlabelled_images"] = len(extra)

In [ ]:
SHARPNESS_SIDE = 512   # focus is measured at a fixed scale so values compare across source sizes
MASK_SIDE = 512

def image_stats(path: str) -> dict:
    """Geometry, exposure, focus and egg-mask statistics for one candled image."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return {"img_ok": False}
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # --- segment the egg: candling makes the egg bright against a dark surround
    small = cv2.GaussianBlur(cv2.resize(gray, (MASK_SIDE, MASK_SIDE), interpolation=cv2.INTER_AREA),
                             (5, 5), 0)
    _, m = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    n_cc, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n_cc > 1:
        k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
        mask = (lab == k).astype(np.uint8)
        x, y, bw, bh, area = st[k, :5]
    else:                                   # degenerate: treat the whole frame as egg
        mask = np.ones_like(small, np.uint8); x = y = 0; bw = bh = MASK_SIDE; area = MASK_SIDE ** 2
    pad = 2
    touches = bool(x <= pad or y <= pad or x + bw >= MASK_SIDE - pad or y + bh >= MASK_SIDE - pad)
    egg_px = small[mask.astype(bool)]

    # --- focus, measured only inside the egg and at a fixed resolution
    fixed = cv2.resize(gray, (SHARPNESS_SIDE, SHARPNESS_SIDE), interpolation=cv2.INTER_AREA)
    fmask = cv2.resize(mask, (SHARPNESS_SIDE, SHARPNESS_SIDE),
                       interpolation=cv2.INTER_NEAREST).astype(bool)
    lap = cv2.Laplacian(fixed, cv2.CV_64F)

    b, g, r = (float(img[:, :, i].mean()) for i in range(3))
    return dict(
        img_ok=True, width=int(w), height=int(h), aspect=float(w / h),
        megapixels=float(w * h / 1e6), is_square=bool(w == h), min_side=int(min(w, h)),
        file_kb=float(Path(path).stat().st_size / 1024),
        mean_lum=float(gray.mean()), std_lum=float(gray.std()),
        p01_lum=float(np.percentile(gray, 1)), p99_lum=float(np.percentile(gray, 99)),
        frac_clipped_hi=float((gray >= 250).mean()), frac_clipped_lo=float((gray <= 5).mean()),
        egg_mean_lum=float(egg_px.mean()) if egg_px.size else np.nan,
        egg_std_lum=float(egg_px.std()) if egg_px.size else np.nan,
        egg_frac_clipped_hi=float((egg_px >= 250).mean()) if egg_px.size else np.nan,
        egg_area_frac=float(area) / (MASK_SIDE ** 2),
        egg_bbox_fill=float(area) / max(float(bw * bh), 1.0),
        egg_touches_border=touches,
        lap_var_global=float(lap.var()),
        lap_var_egg=float(lap[fmask].var()) if fmask.any() else np.nan,
        mean_B=b, mean_G=g, mean_R=r, rb_ratio=float(r / max(b, 1e-6)),
    )

print(f"computing statistics for {len(found)} images with n_jobs={N_JOBS} ...")
rows = Parallel(n_jobs=N_JOBS, verbose=1)(delayed(image_stats)(p) for p in found["path"])
imgstats = pd.concat([found.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
bad = imgstats.loc[~imgstats["img_ok"].fillna(False), "egg_id"].tolist()
print(f"\nunreadable images: {len(bad)}  {bad[:10]}")
imgstats.head(3)

In [ ]:
# --- merge into one working frame ----------------------------------------
df = labels.merge(imgstats.drop(columns=["img_ok"]), on="egg_id", how="left", validate="1:1")
assert len(df) == len(labels)

geom = df[["width", "height", "min_side", "megapixels", "file_kb"]].describe().T.round(1)
savetable(geom, "t10_image_geometry", "Image geometry after square cropping")
print(geom.to_string())
print(f"\nall images square: {bool(df['is_square'].all())}")
print(f"smallest image    : {int(df['min_side'].min())} px  (egg {df.loc[df['min_side'].idxmin(),'egg_id']})")
print(f"largest image     : {int(df['min_side'].max())} px  (egg {df.loc[df['min_side'].idxmax(),'egg_id']})")
print(f"median image      : {int(df['min_side'].median())} px")
SUMMARY["image_min_side"] = {"min": int(df["min_side"].min()), "median": int(df["min_side"].median()),
                             "max": int(df["min_side"].max())}
SUMMARY["all_images_square"] = bool(df["is_square"].all())

In [ ]:
# --- confound screening: do capture conditions differ between classes? ----
CAPTURE_VARS = {
    "min_side": "image size (px)",
    "egg_mean_lum": "mean luminance inside egg",
    "egg_std_lum": "luminance SD inside egg",
    "egg_frac_clipped_hi": "fraction of blown highlights",
    "lap_var_egg": "focus (Laplacian variance)",
    "egg_area_frac": "egg area / frame",
    "rb_ratio": "red/blue ratio (colour temperature)",
}

def kw_by_class(frame, var, by="label4"):
    """Kruskal-Wallis across classes with an epsilon-squared effect size."""
    groups = [g[var].dropna().values for _, g in frame.groupby(by, observed=True)]
    groups = [g for g in groups if len(g)]
    pooled = np.concatenate(groups) if groups else np.array([])
    if len(groups) < 2 or pooled.size == 0 or np.ptp(pooled) == 0:
        # constant or degenerate variable: no difference can exist
        H, p, eps2 = 0.0, 1.0, 0.0
    else:
        H, p = stats.kruskal(*groups)
        n, k = sum(len(g) for g in groups), len(groups)
        eps2 = (H - k + 1) / (n - k) if n > k else np.nan
        if not np.isfinite(p):
            H, p, eps2 = 0.0, 1.0, 0.0
    return {"variable": var, "label": CAPTURE_VARS[var], "H": round(float(H), 2),
            "p_value": float(p), "epsilon_sq": round(float(eps2), 4)}

conf = pd.DataFrame([kw_by_class(df, v) for v in CAPTURE_VARS]).set_index("variable")
# Holm-Bonferroni correction across the simultaneous tests
m = len(conf)
order = conf["p_value"].fillna(1.0).rank(method="first").astype(int)
conf["p_holm"] = np.minimum(1.0, conf["p_value"].fillna(1.0) * (m - order + 1))
conf["significant_after_holm"] = conf["p_holm"] < 0.05
conf["p_value"] = conf["p_value"].round(4); conf["p_holm"] = conf["p_holm"].round(4)
savetable(conf, "t11_capture_confound_screen",
          "Kruskal-Wallis screen for differences in capture conditions between the four classes")
SUMMARY["confound_screen"] = conf.reset_index().to_dict("records")
print(conf.to_string())
print("\nepsilon-squared: <0.01 negligible, 0.01-0.06 small, 0.06-0.14 moderate, >0.14 large")
print("Any variable significant after Holm correction AND with epsilon^2 > 0.06 is a plausible")
print("shortcut feature and must be reported in Section 5.8 and controlled in notebook 03.")
conf

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (var, lab) in zip(axes.ravel(), CAPTURE_VARS.items()):
    data = [df.loc[df["label4"].eq(c), var].dropna().values for c in LABEL4]
    parts = ax.violinplot(data, showmedians=True, widths=0.8)
    for pc, c in zip(parts["bodies"], LABEL4):
        pc.set_facecolor(CFG.PALETTE[c]); pc.set_alpha(0.65)
    for key in ("cmedians", "cbars", "cmins", "cmaxes"):
        if key in parts:
            parts[key].set_color("#333333"); parts[key].set_linewidth(1.0)
    ax.set_xticks(range(1, 5)); ax.set_xticklabels(LABEL4, rotation=20, fontsize=8)
    row = conf.loc[var]
    flag = "  <-- CHECK" if (row["significant_after_holm"] and row["epsilon_sq"] > 0.06) else ""
    ax.set_title(f"{lab}\np(Holm) = {row['p_holm']:.3f}, $\\epsilon^2$ = {row['epsilon_sq']:.3f}{flag}",
                 fontsize=9)
axes.ravel()[-1].axis("off")
fig.suptitle("Capture-condition screening: are any classes separable by acquisition artefacts alone?",
             y=1.01, fontsize=12)
fig.tight_layout()
savefig(fig, "capture_confound_screen",
        "Distribution of capture variables by class, screening for shortcut features")
plt.show()

In [ ]:
# --- egg framing and partial eggs ----------------------------------------
df["partial_egg"] = df["egg_touches_border"]
part = pd.crosstab(df["label4"], df["partial_egg"]).rename(columns={False: "fully_framed",
                                                                   True: "touches_border"})
part["pct_touching"] = (part.get("touches_border", 0) / part.sum(1) * 100).round(1)
savetable(part, "t12_partial_eggs",
          "Eggs whose segmented region touches the frame border (partial / tightly cropped eggs)")
print(part.to_string())
n_part = int(df["partial_egg"].sum())
SUMMARY["n_partial_eggs"] = n_part
print(f"\n{n_part} of {len(df)} eggs ({n_part/len(df)*100:.1f}%) touch the frame border.")
print("These constrain the augmentation policy in notebook 03: random cropping risks removing")
print("crack evidence on these eggs, so padding-safe transforms are preferred.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
for c in LABEL4:
    ax.hist(df.loc[df["label4"].eq(c), "egg_area_frac"], bins=30, alpha=0.55,
            color=CFG.PALETTE[c], label=c)
ax.set_xlabel("egg area / frame area"); ax.set_ylabel("eggs")
ax.set_title("(a) How tightly the crop frames the egg"); ax.legend(fontsize=8)

ax = axes[1]
pv = part.drop(columns="pct_touching").reindex(LABEL4).fillna(0)
bottom = np.zeros(len(pv))
for col, col_c in zip(pv.columns, ["#9ecae1", "#e6550d"]):
    ax.bar(pv.index.astype(str), pv[col], bottom=bottom, label=str(col), color=col_c, width=0.65)
    bottom += pv[col].values
ax.set_ylabel("eggs"); ax.set_title("(b) Partial / border-touching eggs by class")
ax.legend(fontsize=8); ax.tick_params(axis="x", labelrotation=15)
fig.tight_layout()
savefig(fig, "egg_framing", "Egg framing within the square crop and incidence of border-touching eggs")
plt.show()

---
## 4b. Provenance: cropping method and manual review flags

The pipeline that produced these images is itself part of the dataset's description, and Chapter 5
must state it precisely. Three provenance files are reconciled here:

* `mapping.csv` — original capture filename to `E###` identifier
* `crop_log.csv` — output of `crop_eggs.py`, the automatic square crop with 10 % margin
* `review_log.csv` — output of `review_crop.py`, covering the 134 images cropped semi-manually,
  some flagged `PARTIAL` where part of the egg falls outside the frame

Two questions are answered. First, **is the semi-manually cropped subset distributed evenly across
the four classes?** If the difficult crops concentrated in one class, crop method becomes a
confound. Second, **does the automatic egg-segmentation detector built in Section 4 agree with the
human `PARTIAL` flags?** Agreement validates the detector; disagreement tells us which signal to
trust when setting the augmentation policy in notebook 03.

In [ ]:
# --- how the provenance columns are named in YOUR files -------------------
# The notebook auto-detects these, but override here if detection reports the wrong column.
PROVENANCE_HINTS = {
    "egg_id_cols":  ["egg_id", "new_name", "new_filename", "filename", "file", "image", "name"],
    "crop_method_cols": ["method", "crop_method", "mode", "type", "source"],
    "flag_cols":    ["flag", "status", "review_flag", "result", "note", "notes", "comment"],
}

def _read_optional(path: Path, label: str):
    if not path.is_file():
        print(f"  [skip ] {label}: not found at {path}")
        return None
    d = pd.read_csv(path, encoding="utf-8-sig")
    print(f"  [found] {label}: {d.shape[0]} rows, columns = {list(d.columns)}")
    return d

def _pick(df_, candidates):
    """First column whose lowercase name matches a candidate."""
    low = {c.lower().strip(): c for c in df_.columns}
    for cand in candidates:
        if cand in low:
            return low[cand]
    return None

def _extract_egg_id(series):
    return series.astype(str).str.extract(r"(E\d{3})", expand=False)

print("reading provenance files ...")
mapping_df = _read_optional(CFG.MAPPING_CSV, "mapping.csv")
crop_df    = _read_optional(CFG.CROP_LOG_CSV, "crop_log.csv")
review_df  = _read_optional(CFG.REVIEW_LOG_CSV, "review_log.csv")

In [ ]:
# --- fold provenance into the working frame ------------------------------
df["crop_method"] = "automatic"          # default: crop_eggs.py
df["review_flag"] = pd.NA
df["manual_partial"] = False

if review_df is not None and len(review_df):
    idc = _pick(review_df, PROVENANCE_HINTS["egg_id_cols"]) or review_df.columns[0]
    rv = review_df.copy()
    rv["egg_id"] = _extract_egg_id(rv[idc])
    rv = rv.dropna(subset=["egg_id"]).drop_duplicates("egg_id")
    print(f"review_log: matched {rv['egg_id'].notna().sum()} egg ids using column '{idc}'")

    flagc = _pick(rv, PROVENANCE_HINTS["flag_cols"])
    if flagc is None:
        # fall back: any column that contains the token PARTIAL anywhere
        cand = [c for c in rv.columns
                if rv[c].astype(str).str.upper().str.contains("PARTIAL", na=False).any()]
        flagc = cand[0] if cand else None
    print(f"review_log: using '{flagc}' as the flag column")

    rmap = rv.set_index("egg_id")
    df["in_review_log"] = df["egg_id"].isin(rmap.index)
    df.loc[df["in_review_log"], "crop_method"] = "semi-manual"
    if flagc is not None:
        flags = df["egg_id"].map(rmap[flagc])
        df["review_flag"] = flags
        df["manual_partial"] = flags.astype(str).str.upper().str.contains("PARTIAL", na=False)
else:
    df["in_review_log"] = False

n_semi = int((df["crop_method"] == "semi-manual").sum())
n_manual_partial = int(df["manual_partial"].sum())
print(f"\nsemi-manually cropped : {n_semi}  (expected about 134)")
print(f"flagged PARTIAL by hand: {n_manual_partial}")
SUMMARY["n_semi_manual_crops"] = n_semi
SUMMARY["n_manual_partial"] = n_manual_partial

# Q1: is the difficult-crop subset evenly spread across classes?
if n_semi:
    cm = pd.crosstab(df["label4"], df["crop_method"])
    cm_pct = (cm.div(cm.sum(1), axis=0) * 100).round(1)
    chi2_c, p_c, dof_c, exp_c = stats.chi2_contingency(cm)
    savetable(cm, "t16_crop_method_by_class",
              "Cropping method (automatic vs semi-manual) by class")
    SUMMARY["crop_method_class_p"] = float(p_c)
    print(f"\ncrop method x class: chi2({dof_c}) = {chi2_c:.2f}, p = {p_c:.4f}"
          f"  (min expected {exp_c.min():.1f})")
    print(cm.to_string()); print(); print(cm_pct.to_string())

In [ ]:
# --- Q2: does the automatic egg detector agree with the human PARTIAL flags?
if n_manual_partial:
    agree = pd.crosstab(df["manual_partial"].rename("human_PARTIAL"),
                        df["egg_touches_border"].rename("auto_touches_border"))
    tp = int(agree.get(True, pd.Series(dtype=int)).get(True, 0))
    fn = int(agree.get(False, pd.Series(dtype=int)).get(True, 0))
    fp = int(agree.get(True, pd.Series(dtype=int)).get(False, 0))
    tn = int(agree.get(False, pd.Series(dtype=int)).get(False, 0))
    recall = tp / max(tp + fn, 1)
    prec = tp / max(tp + fp, 1)
    kappa = (lambda po, pe: (po - pe) / (1 - pe) if pe < 1 else np.nan)(
        (tp + tn) / len(df),
        ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / len(df) ** 2)
    savetable(agree, "t17_partial_detector_agreement",
              "Agreement between the automatic border-touch detector and the manual PARTIAL flags")
    print(agree.to_string())
    print(f"\nautomatic detector vs human PARTIAL flag:")
    print(f"  recall    {recall:.3f}   (human PARTIAL eggs the detector also caught)")
    print(f"  precision {prec:.3f}")
    print(f"  Cohen's k {kappa:.3f}")
    SUMMARY["partial_detector"] = {"recall": round(recall, 3), "precision": round(prec, 3),
                                   "kappa": round(float(kappa), 3)}
    print("\nThe HUMAN flag is authoritative. The automatic detector is reported only as a")
    print("reproducibility check; notebook 03 excludes human-flagged PARTIAL eggs from any")
    print("transform that could crop away shell area.")
else:
    print("No manual PARTIAL flags available; relying on the automatic detector alone.")
    print("State this in Section 5.3 as a provenance limitation.")

# --- mapping.csv is used only to confirm capture order is recoverable -----
if mapping_df is not None:
    idc = _pick(mapping_df, PROVENANCE_HINTS["egg_id_cols"]) or mapping_df.columns[-1]
    mp = mapping_df.copy()
    mp["egg_id"] = _extract_egg_id(mp[idc])
    matched = mp["egg_id"].notna().sum()
    print(f"\nmapping.csv: {matched}/{len(mp)} rows resolve to an egg id via column '{idc}'")
    print("Capture order is therefore recoverable, which supports the claim in Section 3.7")
    print("that eggs were renamed systematically rather than arbitrarily.")
    SUMMARY["mapping_rows_resolved"] = int(matched)

---
## 5. Crack geometry: are the three morphology classes real?

The morphology taxonomy (Hairline / Star / Network) is the central claim of this research, and a
reviewer is entitled to ask whether it is a genuine structural distinction or three names applied
to one phenomenon. The 335 annotation polygons give an independent, image-free way to answer that:
if the classes are real, they should separate on geometry alone — elongation, area, fragment count
and spatial spread — without any deep learning.

Expected signature from the annotation guideline:

* **Hairline** — one long, thin, highly elongated fracture line
* **Star** — several short radiating fragments from a single impact point: more polygons, higher spatial spread
* **Network** — one large, low-elongation region of diffuse crazing covering a wide area

This section tests those expectations quantitatively and produces the geometric baseline that
notebook 04 will use.

In [ ]:
def parse_polygons(cell):
    """Label Studio polygons -> list of (N,2) arrays in percent units + source size."""
    if not isinstance(cell, str) or not cell.strip():
        return [], (np.nan, np.nan)
    objs = json.loads(cell)
    polys = [np.asarray(o["points"], dtype=float) for o in objs if len(o.get("points", [])) >= 3]
    return polys, (objs[0].get("original_width", np.nan), objs[0].get("original_height", np.nan))

def _shoelace(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def polygon_metrics(cell) -> dict:
    """Resolution-independent geometry of the annotated crack region(s) for one egg."""
    polys, (w, h) = parse_polygons(cell)
    if not polys:
        return {}
    areas, perims, elong, band, cents = [], [], [], [], []
    for p in polys:
        a = _shoelace(p)                                            # percent^2
        per = float(np.sum(np.linalg.norm(np.diff(np.vstack([p, p[:1]]), axis=0), axis=1)))
        pix = np.column_stack([p[:, 0] / 100.0 * w, p[:, 1] / 100.0 * h]).astype(np.float32)
        (_, _), (rw, rh), _ = cv2.minAreaRect(pix)
        lo, hi = sorted([float(rw), float(rh)])
        areas.append(a); perims.append(per)
        elong.append(hi / lo if lo > 1e-6 else np.nan)
        band.append(lo)                                             # short side of the crack band, px
        cents.append(p.mean(axis=0))
    cents = np.asarray(cents)
    return dict(
        poly_n=len(polys), poly_src_w=float(w), poly_src_h=float(h),
        poly_vertices=int(sum(len(p) for p in polys)),
        poly_area_pct=float(np.sum(areas)) / 1e4 * 100,             # % of frame area
        poly_area_largest_pct=float(np.max(areas)) / 1e4 * 100,
        poly_perimeter_pct=float(np.sum(perims)),
        poly_compactness=float(4 * np.pi * np.sum(areas) / max(np.sum(perims) ** 2, 1e-9)),
        poly_elongation_max=float(np.nanmax(elong)),
        poly_elongation_mean=float(np.nanmean(elong)),
        poly_band_px_min=float(np.nanmin(band)),
        poly_band_frac_min=float(np.nanmin(band)) / max(min(w, h), 1),
        poly_centroid_x=float(cents[:, 0].mean()), poly_centroid_y=float(cents[:, 1].mean()),
        poly_spread=float(np.hypot(*cents.std(axis=0))) if len(cents) > 1 else 0.0,
    )

polym = df["crack_region"].apply(polygon_metrics).apply(pd.Series)
df = pd.concat([df, polym], axis=1)
GEOM_COLS = [c for c in polym.columns if c.startswith("poly_") and c not in
             ("poly_src_w", "poly_src_h")]
print(f"geometry computed for {int(df['poly_n'].notna().sum())} cracked eggs, "
      f"{len(GEOM_COLS)} features")

# consistency: does the polygon's recorded source size match the file on disk?
mism = df.dropna(subset=["poly_src_w"]).query("poly_src_w != width")
print(f"polygons whose recorded source size differs from the image file: {len(mism)}")
print("  (polygon points are stored in percent units, so they remain valid at any size; a")
print("   mismatch only means Label Studio saw a differently scaled copy of the same crop.)")
if len(mism):
    print(mism[["egg_id", "poly_src_w", "width"]].head(10).to_string(index=False))
SUMMARY["n_polygon_size_mismatch"] = int(len(mism))

In [ ]:
KEY_GEOM = ["poly_n", "poly_area_pct", "poly_elongation_max", "poly_compactness",
            "poly_spread", "poly_band_frac_min", "poly_vertices"]
gsum = (df.dropna(subset=["crack_pattern"])
          .groupby("crack_pattern_cat", observed=True)[KEY_GEOM]
          .agg(["median", "mean", "std"]).round(3))
savetable(gsum, "t13_crack_geometry_by_pattern",
          "Annotated crack geometry by morphology class")
print(gsum.to_string())

# Kruskal-Wallis per feature across the three patterns
grows = []
for v in KEY_GEOM:
    gs = [g[v].dropna().values for _, g in
          df.dropna(subset=["crack_pattern"]).groupby("crack_pattern_cat", observed=True)]
    H, p = stats.kruskal(*gs)
    n, k = sum(len(g) for g in gs), len(gs)
    grows.append({"feature": v, "H": round(float(H), 1), "p_value": float(p),
                  "epsilon_sq": round(float((H - k + 1) / (n - k)), 3)})
gtest = pd.DataFrame(grows).set_index("feature").sort_values("epsilon_sq", ascending=False)
gtest["p_value"] = gtest["p_value"].apply(lambda p: f"{p:.2e}")
savetable(gtest, "t14_geometry_discrimination",
          "Kruskal-Wallis discrimination of morphology classes by annotated geometry")
SUMMARY["geometry_discrimination"] = gtest.reset_index().to_dict("records")
gtest

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.9))
plots = [("poly_elongation_max", "max elongation (aspect of crack band)", True),
         ("poly_area_pct", "annotated area (% of frame)", True),
         ("poly_n", "number of separate crack polygons", False),
         ("poly_spread", "spatial spread of fragments (% of frame)", False)]
for ax, (var, lab, logy) in zip(axes, plots):
    data = [df.loc[df["crack_pattern"].eq(p), var].dropna().values for p in PATTERNS]
    bp = boxplot(ax, data, PATTERNS, patch_artist=True, widths=0.55, showfliers=False)
    for patch, p in zip(bp["boxes"], PATTERNS):
        patch.set_facecolor(CFG.PALETTE[p]); patch.set_alpha(0.75)
    for i, d in enumerate(data, start=1):
        ax.scatter(np.random.normal(i, 0.055, len(d)), d, s=6, alpha=0.35, color="#333333", zorder=3)
    if logy:
        ax.set_yscale("log")
    ax.set_ylabel(lab, fontsize=9)
    ax.set_title(f"$\\epsilon^2$ = {gtest.loc[var, 'epsilon_sq']:.2f}", fontsize=10)
fig.suptitle("The three morphology classes separate on annotated geometry alone",
             y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "crack_geometry_by_pattern",
        "Annotated crack geometry by morphology class, showing that the taxonomy is structurally grounded")
plt.show()

In [ ]:
# --- geometry-only classifier: how separable are the classes without pixels?
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix,
                             balanced_accuracy_score, f1_score)

gdf = df.dropna(subset=["crack_pattern"]).copy()
Xg = gdf[KEY_GEOM].to_numpy(dtype=float)
yg = gdf["crack_pattern"].to_numpy()

clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=4000, class_weight="balanced",
                                       random_state=CFG.RANDOM_SEED))
cv = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.RANDOM_SEED)
yp = cross_val_predict(clf, Xg, yg, cv=cv)

acc = float((yp == yg).mean())
bacc = float(balanced_accuracy_score(yg, yp))
mf1 = float(f1_score(yg, yp, average="macro"))
print(f"geometry-only 3-class morphology classifier, {CFG.N_SPLITS}-fold CV")
print(f"  accuracy          {acc:.3f}")
print(f"  balanced accuracy {bacc:.3f}")
print(f"  macro F1          {mf1:.3f}")
print(f"  chance (majority) {gdf['crack_pattern'].value_counts(normalize=True).max():.3f}\n")
print(classification_report(yg, yp, digits=3))
SUMMARY["geometry_only_classifier"] = {"accuracy": round(acc, 3),
                                       "balanced_accuracy": round(bacc, 3),
                                       "macro_f1": round(mf1, 3)}

cm = confusion_matrix(yg, yp, labels=PATTERNS)
fig, ax = plt.subplots(figsize=(4.4, 3.8))
im = ax.imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() * 0.6 else "black", fontweight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(PATTERNS); ax.set_yticks(range(3))
ax.set_yticklabels(PATTERNS); ax.set_xlabel("predicted"); ax.set_ylabel("annotated")
ax.set_title(f"Geometry-only baseline\nmacro F1 = {mf1:.3f}"); ax.grid(False)
fig.tight_layout()
savefig(fig, "geometry_only_baseline",
        "Confusion matrix of a logistic-regression classifier using annotated crack geometry only")
plt.show()
print("\nNOTE: this uses the human polygons, so it is an upper reference, not a deployable model.")
print("It establishes that the taxonomy is structurally real. The deployable baselines are in nb 04.")

In [ ]:
# --- where on the egg do cracks occur? -----------------------------------
fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
cr = df.dropna(subset=["crack_pattern"])
for ax, p in zip(axes[:3], PATTERNS):
    s = cr[cr["crack_pattern"].eq(p)]
    ax.scatter(s["poly_centroid_x"], s["poly_centroid_y"], s=26, alpha=0.6,
               color=CFG.PALETTE[p], edgecolor="white", linewidth=0.4)
    ax.add_patch(plt.Circle((50, 50), 35, fill=False, ls="--", lw=1, color="#888888"))
    ax.set_xlim(0, 100); ax.set_ylim(100, 0); ax.set_aspect("equal")
    ax.set_title(f"{p}  (n = {len(s)})"); ax.set_xlabel("x (% of width)")
    if ax is axes[0]:
        ax.set_ylabel("y (% of height)")

ax = axes[3]
h2, xe, ye = np.histogram2d(cr["poly_centroid_x"], cr["poly_centroid_y"],
                            bins=18, range=[[0, 100], [0, 100]])
im = ax.imshow(h2.T, origin="upper", extent=[0, 100, 100, 0], cmap="magma")
ax.set_title("All cracks: spatial density"); ax.set_aspect("equal"); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="eggs")
fig.suptitle("Crack centroid position within the square crop", y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "crack_spatial_distribution",
        "Spatial distribution of annotated crack centroids by morphology class")
plt.show()

# is crack position class dependent? (matters for whether translation augmentation is safe)
for coord in ("poly_centroid_x", "poly_centroid_y"):
    H, p = stats.kruskal(*[cr.loc[cr["crack_pattern"].eq(q), coord].values for q in PATTERNS])
    print(f"{coord}: Kruskal-Wallis H = {H:.2f}, p = {p:.3f}")

---
## 6. Resolution sensitivity: choosing the network input size

Standard ImageNet transfer learning resizes to 224x224. For micro-cracks that is a design risk:
these images are 1300–3500 px square, so 224 discards roughly 95 % of the linear detail. This
section measures the loss empirically rather than assuming it, by cropping the annotated crack
region at native resolution, downsampling to each candidate input size, restoring it, and
measuring how much high-frequency (gradient) energy survives. The chosen input size is defended in
Chapter 4 with this evidence.

In [ ]:
CANDIDATE_SIZES = [224, 288, 320, 384, 448, 512]

def crack_bbox_pct(cell):
    polys, _ = parse_polygons(cell)
    if not polys:
        return None
    allp = np.vstack(polys)
    return allp[:, 0].min(), allp[:, 1].min(), allp[:, 0].max(), allp[:, 1].max()

def retention_for_egg(path, cell, sizes=CANDIDATE_SIZES):
    """Gradient-energy retained inside the annotated crack region after down/up-sampling."""
    bb = crack_bbox_pct(cell)
    if bb is None:
        return None
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    H, W = img.shape
    x0, y0, x1, y1 = [int(round(v / 100.0 * (W if i % 2 == 0 else H))) for i, v in enumerate(bb)]
    x0, y0 = max(x0, 0), max(y0, 0)
    x1, y1 = min(max(x1, x0 + 8), W), min(max(y1, y0 + 8), H)
    ref_crop_scale = (x1 - x0) / W                 # crop width as a fraction of the frame
    ref = img[y0:y1, x0:x1].astype(np.float32)
    e_ref = float(np.mean(cv2.Laplacian(ref, cv2.CV_32F) ** 2))
    out = {"egg_id": Path(path).stem}
    for s in sizes:
        # simulate the whole-frame resize, then look at the same region
        small = cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA)
        back = cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR)
        got = back[y0:y1, x0:x1].astype(np.float32)
        e = float(np.mean(cv2.Laplacian(got, cv2.CV_32F) ** 2))
        out[f"retain_{s}"] = e / e_ref if e_ref > 0 else np.nan
        out[f"psnr_{s}"] = float(cv2.PSNR(ref.astype(np.uint8), got.astype(np.uint8)))
    out["crop_frac"] = ref_crop_scale
    return out

cracked = df.dropna(subset=["crack_pattern"])
print(f"resolution sensitivity over {len(cracked)} cracked eggs ...")
rets = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(retention_for_egg)(r.path, r.crack_region) for r in cracked.itertuples())
ret = pd.DataFrame([r for r in rets if r]).merge(
    df[["egg_id", "crack_pattern"]], on="egg_id", how="left")

retain_cols = [f"retain_{s}" for s in CANDIDATE_SIZES]
ret_sum = ret.groupby("crack_pattern", observed=True)[retain_cols].median().round(3)
ret_sum.loc["ALL"] = ret[retain_cols].median().round(3)
ret_sum.columns = [f"{s}px" for s in CANDIDATE_SIZES]
savetable(ret_sum, "t15_resolution_retention",
          "Median high-frequency energy retained inside the annotated crack region at each input size")
SUMMARY["resolution_retention_median"] = ret[retain_cols].median().round(3).to_dict()
print(ret_sum.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
ax = axes[0]
for p in PATTERNS:
    s = ret[ret["crack_pattern"].eq(p)]
    med = [s[f"retain_{z}"].median() for z in CANDIDATE_SIZES]
    q1 = [s[f"retain_{z}"].quantile(0.25) for z in CANDIDATE_SIZES]
    q3 = [s[f"retain_{z}"].quantile(0.75) for z in CANDIDATE_SIZES]
    ax.plot(CANDIDATE_SIZES, med, "o-", color=CFG.PALETTE[p], label=p, lw=1.8)
    ax.fill_between(CANDIDATE_SIZES, q1, q3, color=CFG.PALETTE[p], alpha=0.15)
ax.axhline(0.5, ls="--", lw=1, color="#888888")
ax.text(CANDIDATE_SIZES[0], 0.52, "50 % of crack detail retained", fontsize=8, color="#666666")
ax.set_xlabel("network input size (px)"); ax.set_ylabel("gradient energy retained (ratio)")
ax.set_title("(a) Crack detail surviving the resize"); ax.legend(fontsize=9)
ax.set_xticks(CANDIDATE_SIZES)

ax = axes[1]
for p in PATTERNS:
    s = ret[ret["crack_pattern"].eq(p)]
    ax.plot(CANDIDATE_SIZES, [s[f"psnr_{z}"].median() for z in CANDIDATE_SIZES],
            "s-", color=CFG.PALETTE[p], label=p, lw=1.8)
ax.set_xlabel("network input size (px)"); ax.set_ylabel("PSNR of the crack region (dB)")
ax.set_title("(b) Reconstruction fidelity of the crack region"); ax.legend(fontsize=9)
ax.set_xticks(CANDIDATE_SIZES)
fig.suptitle("Empirical justification for the input resolution", y=1.02, fontsize=12)
fig.tight_layout()
savefig(fig, "resolution_sensitivity",
        "Retention of crack detail at candidate network input sizes, used to justify the pre-processing resize")
plt.show()

print("Read the curve, do not assume: the input size chosen in notebook 02 is the smallest size")
print("at which the median retention curve has flattened, traded against GPU memory and the")
print("pretrained weights available at that resolution.")

---
## 7. Visual reference grids

Figures for Chapter 4. Each grid shows representative eggs per class with the annotation polygon
overlaid, so the reader can see what distinguishes the morphologies.

In [ ]:
def load_rgb(path, max_side=900):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    h, w = img.shape[:2]
    if max(h, w) > max_side:
        sc = max_side / max(h, w)
        img = cv2.resize(img, (int(w * sc), int(h * sc)), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def draw_polys(ax, cell, shape):
    polys, _ = parse_polygons(cell)
    h, w = shape[:2]
    for p in polys:
        xy = np.column_stack([p[:, 0] / 100.0 * w, p[:, 1] / 100.0 * h])
        ax.add_patch(plt.Polygon(xy, closed=True, fill=False, edgecolor="#00E5FF",
                                 lw=1.6, alpha=0.95))

def sample_grid(frame, title, n=SAMPLE_GRID_N, seed=CFG.RANDOM_SEED, overlay=True):
    s = frame.sample(min(n, len(frame)), random_state=seed)
    cols = 4; rows = int(np.ceil(len(s) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 3.25 * rows))
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), s.iterrows()):
        img = load_rgb(r["path"])
        if img is None:
            ax.axis("off"); continue
        ax.imshow(img)
        if overlay:
            draw_polys(ax, r["crack_region"], img.shape)
        ax.set_title(f"{r['egg_id']} | {r['age_class']}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for ax in np.atleast_1d(axes).ravel()[len(s):]:
        ax.axis("off")
    fig.suptitle(title, y=1.0, fontsize=12)
    fig.tight_layout()
    return fig

for cls in LABEL4:
    sub_c = df[df["label4"].eq(cls)]
    fig = sample_grid(sub_c, f"{cls}  (n = {len(sub_c)})", overlay=(cls != "Intact"))
    savefig(fig, f"samples_{cls.lower()}",
            f"Representative candled images of class {cls} with annotation polygons overlaid",
            dpi=150)
    plt.show()

In [ ]:
# --- extremes: the eggs that will be hardest for the model ---------------
hardest = pd.concat([
    df.nlargest(4, "lead_time").assign(why="longest to annotate"),
    df.dropna(subset=["crack_pattern"]).nsmallest(4, "poly_area_pct").assign(why="smallest crack"),
])
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, (_, r) in zip(axes.ravel(), hardest.iterrows()):
    img = load_rgb(r["path"])
    if img is None:
        ax.axis("off"); continue
    ax.imshow(img); draw_polys(ax, r["crack_region"], img.shape)
    ax.set_title(f"{r['egg_id']} | {r['label4']}\n{r['why']} ({r['lead_time']:.0f}s)", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle("Difficult cases: longest annotation times and smallest annotated cracks",
             y=1.0, fontsize=12)
fig.tight_layout()
savefig(fig, "difficult_cases",
        "Eggs requiring the longest annotation time and those with the smallest annotated crack area",
        dpi=150)
plt.show()

---
## 8. Export: the master table

Every later notebook reads `data/processed/master_table.csv` and nothing else. It is the joined,
validated, feature-enriched record of all 652 eggs. Writing it here means the split in notebook 02
and every experiment after it operate on an identical, auditable table.

In [ ]:
# store the FILE NAME, not the absolute path, so the table is portable between the
# training machine and the laptop and is safe to commit to GitHub
df["file_name"] = df["path"].apply(lambda p: Path(p).name)

MASTER_COLS = (
    ["egg_id", "file_name", "egg_status", "crack_pattern", "label4", "is_cracked", "age_class",
     "physiological_nutritional_pct", "mechanical_environmental_pct",
     "interpretation", "suggested_actions",
     "annotation_id", "id", "annotator", "lead_time", "created_at", "updated_at",
     "width", "height", "min_side", "megapixels", "file_kb", "is_square",
     "mean_lum", "std_lum", "p01_lum", "p99_lum", "frac_clipped_hi", "frac_clipped_lo",
     "egg_mean_lum", "egg_std_lum", "egg_frac_clipped_hi", "egg_area_frac", "egg_bbox_fill",
     "egg_touches_border", "partial_egg", "crop_method", "review_flag", "manual_partial",
     "in_review_log", "lap_var_global", "lap_var_egg",
     "mean_B", "mean_G", "mean_R", "rb_ratio", "n_polygons", "crack_region"]
    + GEOM_COLS
)
MASTER_COLS = [c for c in dict.fromkeys(MASTER_COLS) if c in df.columns]
master = df[MASTER_COLS].copy()
master["label4_idx"] = master["label4"].map(CFG.LABEL4_TO_IDX).astype("Int64")
master["age_idx"] = master["age_class"].map(CFG.AGE_TO_IDX).astype("Int64")

mpath = PROC_DIR / "master_table.csv"
master.to_csv(mpath, index=False)
print(f"wrote {mpath}  ({master.shape[0]} rows x {master.shape[1]} columns)")
print("  images are referenced by file_name only; later notebooks resolve them as")
print("  CFG.IMAGES_DIR / file_name, so this file works on any machine.")

# also persist the recovered rule base in the dashboard's format
rc = rules[["crack_pattern", "age_class", "phys_pct", "mech_pct"]].copy()
rc = rc.merge(sub.groupby(["crack_pattern_cat", "age_class"], observed=True)
                 .agg(interpretation=("interpretation", "first"),
                      suggested_actions=("suggested_actions", "first")).reset_index()
                 .rename(columns={"crack_pattern_cat": "crack_pattern"}),
              on=["crack_pattern", "age_class"], how="left")
rc.to_csv(PROC_DIR / "root_cause_rules_recovered.csv", index=False)
print(f"wrote {PROC_DIR / 'root_cause_rules_recovered.csv'}  ({len(rc)} rules)")
master.head(3)

In [ ]:
# --- machine-readable digest + figure/table index -------------------------
SUMMARY["figures"] = FIGURE_INDEX
SUMMARY["tables"] = TABLE_INDEX
SUMMARY["master_table"] = {"path": str(mpath), "rows": int(master.shape[0]),
                           "cols": int(master.shape[1])}

def _json_safe(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (np.ndarray,)): return o.tolist()
    return str(o)

spath = OUT / "eda_summary.json"
spath.write_text(json.dumps(SUMMARY, indent=2, default=_json_safe))
print(f"wrote {spath}")

pd.DataFrame(FIGURE_INDEX).to_csv(OUT / "figure_index.csv", index=False)
pd.DataFrame(TABLE_INDEX).to_csv(OUT / "table_index.csv", index=False)
print(f"{len(FIGURE_INDEX)} figures, {len(TABLE_INDEX)} tables recorded")

In [ ]:
# =========================================================================
# DIGEST -- copy this block back to your supervisor / for the next stage
# =========================================================================
L = []
A = L.append
A("=" * 70); A("NOTEBOOK 01 DIGEST"); A("=" * 70)
A(f"generated            : {SUMMARY['generated_at']}")
A(f"eggs                 : {SUMMARY['n_eggs']}   image files: {SUMMARY['n_image_files']}")
A(f"integrity checks     : {SUMMARY['integrity_checks_total'] - SUMMARY['integrity_checks_failed']}"
  f"/{SUMMARY['integrity_checks_total']} passed")
A(f"class counts         : {SUMMARY['class_counts_label4']}")
A(f"imbalance ratio      : {SUMMARY['imbalance_ratio_label4']}")
A(f"age counts           : {SUMMARY['class_counts_age']}")
A(f"image size (min side): {SUMMARY['image_min_side']}")
A(f"all square           : {SUMMARY['all_images_square']}")
A(f"partial eggs (auto)  : {SUMMARY['n_partial_eggs']}")
A(f"semi-manual crops    : {SUMMARY.get('n_semi_manual_crops', 'n/a')}   "
  f"human PARTIAL flags: {SUMMARY.get('n_manual_partial', 'n/a')}")
A(f"crop method x class p: {SUMMARY.get('crop_method_class_p', 'n/a')}")
A(f"partial detector     : {SUMMARY.get('partial_detector', 'n/a')}")
A(f"rule base determinist: {SUMMARY['root_cause_rules_deterministic']}")
A("")
A("-- association of labels with breeder age class --")
for r in SUMMARY["association_tests"]:
    A(f"   {r['comparison']:<48s} chi2={r['chi2']:>7.3f} p={r['p_value']:<7.4f} V={r['cramers_v']}")
A("")
A("-- capture-condition confound screen (Holm corrected) --")
for r in SUMMARY["confound_screen"]:
    flag = " *** SHORTCUT RISK" if (r["significant_after_holm"] and r["epsilon_sq"] > 0.06) else ""
    A(f"   {r['label']:<34s} p_holm={r['p_holm']:<7.4f} eps2={r['epsilon_sq']:<7.4f}{flag}")
A("")
A("-- morphology discrimination by annotated geometry --")
for r in SUMMARY["geometry_discrimination"]:
    A(f"   {r['feature']:<24s} eps2={r['epsilon_sq']:<6.3f} p={r['p_value']}")
A(f"   geometry-only classifier: {SUMMARY['geometry_only_classifier']}")
A("")
A("-- crack detail retained after resize --")
for k, v in SUMMARY["resolution_retention_median"].items():
    A(f"   {k:<12s} {v}")
A("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print(digest)

---
## 9. Findings — to be written after the run

> Complete this cell after inspecting the output above. These sentences become Section 5.3
> (Dataset Description) and part of Section 5.8 (Threats to Validity) of the dissertation.

1. **Label integrity:** …
2. **Class structure:** the four-class target is moderately imbalanced (ratio …), with *Star*
   the minority class at … eggs, which motivates … in notebook 05.
3. **Pattern and breeder age are / are not statistically associated** (χ² = …, p = …, V = …).
   Consequence for the architecture: …
4. **Capture conditions:** … of the seven capture variables differed significantly between
   classes after Holm correction, with effect sizes of …, meaning …
5. **Morphology taxonomy:** the three classes separate on annotated geometry alone
   (geometry-only macro F1 = …), confirming the taxonomy is structurally grounded rather than
   nominal.
6. **Resolution:** median crack-detail retention is … at 224 px and … at … px, so the
   pre-processing resize in notebook 02 is set to … px.
7. **Augmentation constraints carried into notebook 03:** … partial eggs, cracks located …,
   therefore …